# Lambdas e referências a métodos

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Passar comportamento como valor com uma lambda ou uma referência a método, como um teste ou um comparador.
- Montar comparadores com Comparator.comparing, thenComparing e reversed.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma lambda: uma função como valor
2. Lambdas no dia a dia
3. Referências a métodos
4. Comparadores: comparing, reversed, thenComparing
5. Passando comportamento para os seus métodos
6. Variáveis dentro de uma lambda

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Uma lambda: uma função como valor

Uma **lambda** é uma pequena função escrita como valor: `n -> n * 2` recebe `n` e devolve `n * 2`. Você pode guardá-la numa variável e passá-la para métodos. O tipo dela diz o que recebe e o que devolve:

| Tipo | Recebe → devolve | Chame com |
|---|---|---|
| `Predicate<T>` | um T → verdadeiro ou falso | `test(x)` |
| `Function<T, R>` | um T → um R | `apply(x)` |
| `Consumer<T>` | um T → nada (ele faz algo) | `accept(x)` |

In [ ]:
Predicate<Integer> isEven = n -> n % 2 == 0;
Function<String, Integer> length = s -> s.length();
Consumer<String> shout = s -> IO.println(s.toUpperCase() + "!");

IO.println(isEven.test(4) + " " + isEven.test(7));
IO.println(length.apply("lambda"));
shout.accept("olá");

🧪 **Experimente:** escreva um `Predicate<String>` que diz se uma palavra começa com `"a"`, e teste em duas palavras.

## 2. Lambdas no dia a dia

Muitos métodos recebem uma lambda para saber **o que fazer**: `removeIf` recebe o teste dos itens a tirar, `forEach` o que fazer com cada item, e `sort` como comparar dois itens. Um comparador recebe dois itens e devolve um número negativo, zero ou positivo. `Comparator.reverseOrder()` e `Comparator.comparing(...)` montam os mais comuns para você.

In [ ]:
List<Integer> numbers = new ArrayList<>(List.of(5, 12, 3, 8, 21, 6));
numbers.removeIf(n -> n % 3 == 0);
IO.println(numbers);
numbers.sort(Comparator.reverseOrder());
IO.println(numbers);
numbers.forEach(n -> IO.print(n * 10 + " "));
IO.println();

🧪 **Experimente:** ordene em ordem crescente com a lambda `(a, b) -> Integer.compare(a, b)`. O que `(a, b) -> Integer.compare(b, a)` faz? (Evite `a - b`: estoura para números muito distantes.)

## 3. Referências a métodos

Quando uma lambda só chama um método que já existe, você pode citar o método no lugar: `s -> s.toUpperCase()` vira `String::toUpperCase`, e `x -> System.out.println(x)` vira `System.out::println`. É a mesma coisa, mais curta.

In [ ]:
List<String> words = new ArrayList<>(List.of("pera", "figo", "banana"));
words.replaceAll(String::toUpperCase);       // o mesmo que s -> s.toUpperCase()
words.forEach(System.out::println);          // o mesmo que w -> System.out.println(w)
words.sort(Comparator.comparing(String::length));
IO.println(words);

🧪 **Experimente:** reescreva as duas referências a métodos como lambdas, rode e confira que a saída é a mesma.

## 4. Comparadores: comparing, reversed, thenComparing

Escrever comparadores `(a, b) -> …` na mão é fácil de errar. `Comparator.comparing(Player::name)` (ou `comparingInt` para números) monta um a partir do valor a comparar; `.reversed()` o inverte, e `.thenComparing(...)` desempata com um segundo valor. Os slides montam um passo a passo.

#### 🎞️ Montando um comparador

![comparingInt(score) põe as pontuações menores primeiro, reversed() inverte, e thenComparing(name) desempata Bia e Caio, dando Ana 75, Bia 50, Caio 50](comparator-steps.pt-br.svg)

In [ ]:
record Player(String name, int score) {}

void main() {
    List<Player> table = new ArrayList<>(List.of(
            new Player("Caio", 50), new Player("Ana", 75), new Player("Bia", 50)));
    table.sort(Comparator.comparing(Player::name));
    IO.println(table);
    table.sort(Comparator.comparingInt(Player::score).reversed().thenComparing(Player::name));
    IO.println(table);
}

🧪 **Experimente:** ordene pela menor pontuação, desempatando pelo nome em ordem inversa (`Comparator.comparing(Player::name).reversed()` dentro do `thenComparing`).

### ✏️ Exercício: O mais barato primeiro

Ordene os produtos do mais barato; produtos com o mesmo preço vão em ordem alfabética. Use `Comparator.comparingDouble` e `thenComparing`.

In [ ]:
record Product(String name, double price) {}

/** The names after sortCheapestFirst. */
List<String> names(List<Product> items) {
    List<Product> copy = new ArrayList<>(items);
    sortCheapestFirst(copy);
    return copy.stream().map(Product::name).toList();
}

void sortCheapestFirst(List<Product> items) {
    // TODO: items.sort(Comparator.comparingDouble(Product::price)…)
}

void main() {
    check("pelo preço", names(List.of(new Product("pen", 2.5), new Product("book", 30), new Product("gum", 1))), List.of("gum", "pen", "book"));
    check("um empate", names(List.of(new Product("tea", 4), new Product("coffee", 4))), List.of("coffee", "tea"));
}

<details>
<summary>Ver uma resposta</summary>

```java
record Product(String name, double price) {}

/** The names after sortCheapestFirst. */
List<String> names(List<Product> items) {
    List<Product> copy = new ArrayList<>(items);
    sortCheapestFirst(copy);
    return copy.stream().map(Product::name).toList();
}

void sortCheapestFirst(List<Product> items) {
    items.sort(Comparator.comparingDouble(Product::price).thenComparing(Product::name));
}

void main() {
    check("pelo preço", names(List.of(new Product("pen", 2.5), new Product("book", 30), new Product("gum", 1))), List.of("gum", "pen", "book"));
    check("um empate", names(List.of(new Product("tea", 4), new Product("coffee", 4))), List.of("coffee", "tea"));
}
```

</details>

## 5. Passando comportamento para os seus métodos

Os seus métodos também podem receber uma lambda: declare um parâmetro de um tipo funcional, como `Predicate<String>`, e chame o método dele (`test`) lá dentro. Quem chama decide o comportamento, e um método serve muitos casos.

In [ ]:
int countIf(List<String> words, Predicate<String> test) {
    int count = 0;
    for (String w : words) {
        if (test.test(w)) {
            count++;
        }
    }
    return count;
}

void main() {
    List<String> words = List.of("sky", "stream", "lambda", "map");
    IO.println(countIf(words, w -> w.length() <= 3));
    IO.println(countIf(words, w -> w.startsWith("s")));
    IO.println(countIf(words, String::isEmpty));
}

🧪 **Experimente:** conte as palavras que têm um "a". Depois escreva `apply(List<String> words, Function<String, String> f)`, que devolve uma lista nova com `f` aplicada a cada palavra.

### ✏️ Exercício: Fique com as palavras que combinam

Devolva uma lista nova só com as palavras que passam no teste, em ordem, chamando `test.test(palavra)`.

In [ ]:
List<String> keep(List<String> words, Predicate<String> test) {
    // TODO: acrescente cada palavra para a qual test.test(palavra) é verdadeiro
    return List.of();
}

void main() {
    check("palavras curtas", keep(List.of("a", "bcd", "ef"), w -> w.length() < 3), List.of("a", "ef"));
    check("maiúsculas", keep(List.of("Hi", "hey", "Yo"), w -> Character.isUpperCase(w.charAt(0))), List.of("Hi", "Yo"));
    check("nenhuma passa", keep(List.of("x"), w -> false), List.of());
}

<details>
<summary>Ver uma resposta</summary>

```java
List<String> keep(List<String> words, Predicate<String> test) {
    List<String> result = new ArrayList<>();
    for (String w : words) {
        if (test.test(w)) {
            result.add(w);
        }
    }
    return result;
}

void main() {
    check("palavras curtas", keep(List.of("a", "bcd", "ef"), w -> w.length() < 3), List.of("a", "ef"));
    check("maiúsculas", keep(List.of("Hi", "hey", "Yo"), w -> Character.isUpperCase(w.charAt(0))), List.of("Hi", "Yo"));
    check("nenhuma passa", keep(List.of("x"), w -> false), List.of());
}
```

</details>

## 6. Variáveis dentro de uma lambda

Uma lambda pode usar as variáveis locais em volta dela, mas só se elas nunca mudarem depois de receberem o valor ("efetivamente finais"). Isso impede que uma lambda veja um valor que mudou depois que ela foi criada. Para acumular dentro de uma lambda, use um stream (próxima lição) ou um campo em vez de um contador local.

In [ ]:
int limit = 3;                                  // nunca muda: tudo certo
Predicate<String> isShort = w -> w.length() <= limit;
IO.println(isShort.test("map") + " " + isShort.test("stream"));

🧪 **Experimente:** acrescente `limit = 4;` depois da lambda e leia o erro ("must be final or effectively final").

## Em resumo

Uma **lambda** é uma pequena função escrita como valor: `n -> n % 2 == 0` recebe `n` e responde verdadeiro ou falso. Uma **referência a método** como `String::length` cita um método que já existe. Muitos métodos recebem uma delas para saber *o que fazer*: quais itens tirar, ou como comparar dois itens.

```java
record Player(String name, int score) {}

void main() {
    Predicate<Integer> isEven = n -> n % 2 == 0;
    IO.println(isEven.test(4));

List<Player> table = new ArrayList<>(List.of(
            new Player("Caio", 50), new Player("Ana", 75), new Player("Bia", 50)));
    table.sort(Comparator.comparingInt(Player::score).reversed()
            .thenComparing(Player::name));
    IO.println(table);   // Ana 75, depois Bia e Caio, empatados em 50
}
```

Um comparador devolve um número negativo, zero ou positivo, dizendo qual de dois itens vem primeiro.

**Cuidado:** uma lambda só pode usar uma variável local se essa variável nunca mudar depois de receber o valor.

## Leituras

- [Writing Your First Lambda Expression](https://dev.java/learn/language/fp/lambdas/first-lambdas/) (dev.java (Oracle), 9 min): Por que uma lambda precisa de uma interface funcional, os exemplos de Predicate e Consumer, e como a sintaxe fica mais curta passo a passo.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Ordenar com Comparator (lambdas)** · Fácil